# Creación de la vista base

In [ ]:
# src\templates\products\product_base.html
<!DOCTYPE html>
<html>
    <head>
        <title> {% block title %}Ventana de productos{% endblock title %}</title>
        <style>
            .red_line{
                background-color:#FFC7BD;
                align:center;
            }
            .blue_line{
                background-color:#BDF8FF;
                align:center;
            }
        </style>
    </head>
    <body>
        <h1>Ejemplo Ecommerce</h1>
        {% block content %}{% endblock content %}
    </body>
    <footer>
        <p>Ecommerce. SA de CV.</p>
    </footer>
</html>

# Vista heredada para ver lista de productos

In [ ]:
# src\templates\products\product_list.html
{% extends "products/product_base.html" %}
{% block title %}
Lista de productos
{% endblock title %}
{% block content %}
<table>
    {% include "products/product_title_info.html" with titulo="Lista de productos actualizada" horario=consult_time %}
    <thead>
        <tr>
            <th>ID</th>
            <th>Nombre</th>
            <th>Precio</th>
            <th>Producto digital</th>
        </tr>
    </thead>
    <tbody>
        {% for obj in list_prod %}
            <tr 
                {% if forloop.counter|divisibleby:2 %}
                    class="red_line"
                {% else %}
                    class="blue_line"
                {% endif %}>
                <td>
                    {{ obj.id }}
                </td>
                <td>
                    {{ obj.title|upper }}
                </td>
                <td>
                    ${{ obj.price }}
                </td>
                <td>
                    {{ obj.is_digital|yesno}}
                </td>
                
            </tr>
        {% endfor %}
        
    </tbody>
</table>
{% endblock content %}




# Vista de include, para titulos e información

In [ ]:
# src\templates\products\product_title_info.html
<h1>{{ titulo }}</h1>

{% if horario %}
<h2>{{ horario|date:"j/n/Y g:i A"}}</h2>
{% endif %}


# Vista heredada de creación de productos

In [ ]:
# src\templates\products\product_create.html
{% extends "products/product_base.html" %}
{% block title %}
Crear registro de producto
{% endblock title %}
{% block content %}
{% include "products/product_title_info.html" with titulo="Crear productos"%}
<form method="POST" action=""> 
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Crear">
</form>
{% endblock content %}


# Módulo de vistas de products

In [ ]:
# src\products\views.py
from django.shortcuts import render
from datetime import datetime
from products.models import Product
from products.forms import ProductModelForm
# Create your views here.
def list_product(request):
    data = Product.objects.all()
    info = []
    for rec in data:
        info.append({
            "title":rec.title,
            "price":rec.price,
            "is_digital":rec.is_digital,
            "id":rec.id,
        })
    context = {
        "list_prod":info,
        "consult_time":datetime.now().today()
    }
    template = "products/product_list.html"
    return render(request,template,context)

def create_product(request):
    template ="products/product_create.html"
    form = ProductModelForm(request.POST or None)
    if form.is_valid():
        form.save()
    return render(request,template,{"form":form})

# Módulo de forms para vista de creación

In [ ]:
# src\products\forms.py
from django import forms
from .models import Product

class ProductModelForm(forms.ModelForm):
    class Meta:
        model = Product
        fields = [
            "title",
            "slug",
            "description",
            "price",
            "is_digital",
        ]
        exclude = [
            "image",
            "featured",
            "active",
            "tiemstamp",
        ]
        labels = {
            "title":"Agrega el nombre del producto",
            "slug":"Agrega el slug del producto",
            "description":"Añade la descripción del producto",
            "price":"Añade el precio del producto",
            "is_digital":"¿Es producto digital?",
        }

    def clean_title(self, *args, **kwargs):
        title = self.cleaned_data.get("title")
        if len(title)<=10:
            raise forms.ValidationError("Debe de tener más de 10 caracteres")
        return title
    
    def clean_slug(self, *args, **kwargs):
        slug = self.cleaned_data.get("slug")
        if len(slug)<=10:
            raise forms.ValidationError("Debe de tener más de 10 caracteres")
        if " " in slug:
            raise forms.ValidationError("No debe de tener espacios")
        return slug
    
    def clean_price(self, *args, **kwargs):
        price = self.cleaned_data.get("price")
        if price<=0:
            raise forms.ValidationError("El precio debe ser mayor a 0.")
        return price

# Módulo de urls de products

In [ ]:
# src\products\urls.py
from django.urls import path

from .views import list_product, create_product

urlpatterns = [
    path("list", list_product),
    path("create",create_product),
]


# Modificación a urls globales

In [ ]:
# src\config\urls.py
from django.conf import settings
from django.contrib import admin
from django.urls import include, path

urlpatterns = [
    path("up/", include("up.urls")),
    path("", include("pages.urls")),
    path("analytics/", include("analytics.urls")),
    path("users/",include("users.urls")),
    path("products/",include("products.urls")),
    path("admin/", admin.site.urls),
]
if not settings.TESTING:
    urlpatterns = [
        *urlpatterns,
        path("__debug__/", include("debug_toolbar.urls")),
    ]